# Guardrails with LangChain

Guardrails are safety mechanisms that sit between the user and your LLM agent. They intercept the conversation at key points — before the model sees the input, around tool calls, and after the model produces a response — to catch harmful content, leaked PII, and unsafe outputs before they cause damage.

In LangChain, guardrails are implemented as **middleware** passed into `create_agent()`. They execute in order, forming a layered defense stack.

```
User Input
    |
[Layer 1] ContentFilterMiddleware     -- Deterministic input filter
    |
[Layer 2] PIIMiddleware (input)       -- PII redaction on input
    |
[Layer 3] HumanInTheLoopMiddleware    -- Approval for sensitive tools
    |
[Layer 4] PIIMiddleware (output)      -- PII redaction on output
    |
[Layer 5] SafetyGuardrailMiddleware   -- Model-based output safety
    |
User Response
```

There are two fundamental approaches:

- **Deterministic** — regex and keyword rules. Fast, free, but misses nuanced intent.
- **Model-based** — an LLM acts as a safety judge. Understands context, but costs tokens.

The right pattern is to run deterministic checks first (cheap, catches obvious violations) and model-based checks last (expensive, catches subtle ones).

In [10]:
!pip install langchain langchain-groq langgraph python-dotenv -q


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.tools import tool

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

## Layer 1 — Content Filter Middleware

This is the first line of defense. It runs **before** the agent processes anything, so blocked requests never touch the LLM — zero token cost.

We extend `AgentMiddleware` and override the `before_agent()` hook. If a banned keyword is found, we return a canned response and set `jump_to: end` to short-circuit the entire pipeline. The decorator `@hook_config(can_jump_to=["end"])` is required to allow that short-circuit.

This is a deterministic guardrail — simple, fast, and predictable.

In [17]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime

class ContentFilterMiddleware(AgentMiddleware):
    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        first = state["messages"][0] if state["messages"] else None
        if not first or first.type != "human":
            return None
        for kw in self.banned_keywords:
            if kw in first.content.lower():
                return {
                    "messages": [{"role": "assistant", "content": "Request blocked: inappropriate content."}],
                    "jump_to": "end"
                }
        return None  # None means: let the request pass through

## Layer 2 & 4 — PII Middleware

LangChain ships a built-in `PIIMiddleware` that detects and handles Personally Identifiable Information. You configure it with a PII type, a strategy, and whether it applies to input or output.

**Strategies:**
- `redact` — replaces with `[REDACTED_EMAIL]`
- `mask` — replaces with `****-****-****-1234`
- `block` — raises an exception, stopping execution entirely

Layer 2 strips PII from the user's input before it reaches the model. Layer 4 strips any PII that the model might have echoed back in its response. Together they ensure PII never enters your logs or downstream systems.

In [18]:
from langchain.agents.middleware import PIIMiddleware

# Layer 2 — applied to incoming user messages
pii_input_email = PIIMiddleware("email",      strategy="redact", apply_to_input=True)
pii_input_cc    = PIIMiddleware("credit_card", strategy="mask",   apply_to_input=True)

# Layer 4 — applied to the model's outgoing response
pii_output_email = PIIMiddleware("email",     strategy="redact", apply_to_output=True)

## Layer 3 — Human-in-the-Loop Middleware

Some tool calls are too consequential for an AI to execute autonomously — sending emails, deleting records, processing payments. `HumanInTheLoopMiddleware` pauses the agent before those calls and waits for a human decision.

The `interrupt_on` dict maps tool names to a boolean. `True` means pause and wait. `False` means auto-execute.

A **checkpointer** (`InMemorySaver`) is required so the agent can save its state across the interruption and resume exactly where it left off after the human responds.

In [19]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_web(query: str) -> str:
    """Search the web."""
    return f"Results: {query}"

@tool
def send_email(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": True,   # pause and wait for approval
        "search_web": False   # safe to auto-execute
    }
)

## Layer 5 — Model-Based Output Safety

The final layer is a model-based guardrail on the output. Even if a request passes all the input filters, the model could still produce a harmful or inappropriate response. Here we use the LLM itself as a safety judge.

We override the `after_agent()` hook, which runs after the agent produces its response but before the user sees it. The judge is asked a simple yes/no question — `SAFE` or `UNSAFE` — and if it answers `UNSAFE`, the response is replaced with a safe fallback.

Because the judge only needs to answer a binary question, a fast model like Groq is ideal here.

In [20]:
from langchain_core.messages import AIMessage

class SafetyGuardrailMiddleware(AgentMiddleware):
    def __init__(self):
        super().__init__()
        self.judge = ChatGroq(
            model="llama-3.3-70b-versatile",
            api_key=os.getenv("GROQ_API_KEY"),
            temperature=0
        )

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        last = state["messages"][-1] if state["messages"] else None
        if not last or not isinstance(last, AIMessage):
            return None
        verdict = self.judge.invoke([
            {"role": "user", "content": f"Is this response safe? Reply only SAFE or UNSAFE.\n\n{last.content}"}
        ])
        if "UNSAFE" in verdict.content.upper():
            last.content = "I am unable to provide that response. Please rephrase your request."
        return None

## Final Agent — All 5 Layers Combined

Middleware executes in the order it is listed. Input guardrails (Layers 1-2) run first as the request comes in. HITL (Layer 3) wraps tool calls. Output guardrails (Layers 4-5) run last before the response reaches the user.

A `checkpointer` is passed because Layer 3 needs it to persist state across human interruptions.

In [22]:
from langchain.agents import create_agent

class PIIEmailInput(PIIMiddleware):
    def __init__(self): super().__init__("email", strategy="redact", apply_to_input=True)

class PIICCInput(PIIMiddleware):
    def __init__(self): super().__init__("credit_card", strategy="mask", apply_to_input=True)

class PIIEmailOutput(PIIMiddleware):
    def __init__(self): super().__init__("email", strategy="redact", apply_to_output=True)

agent = create_agent(
    model=llm,
    tools=[search_web, send_email],
    middleware=[
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),
        PIIEmailInput(),
        PIICCInput(),
        hitl,
        PIIEmailOutput(),
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("Agent with all 5 guardrail layers ready.")

Agent with all 5 guardrail layers ready.


## Tests

In [24]:
# Layer 1: banned keyword — never hits the LLM
r = agent.invoke(
    {"messages": [{"role": "user", "content": "How do I hack a server?"}]},
    config={"configurable": {"thread_id": "test_1"}}
)
print("Layer 1 (blocked):", r["messages"][-1].content)

Layer 1 (blocked): Request blocked: inappropriate content.


In [25]:
r = agent.invoke(
    {"messages": [{"role": "user", "content": "My email is user@test.com, search for headache remedies"}]},
    config={"configurable": {"thread_id": "test_2"}}
)
print("Layer 2 (PII redacted):", r["messages"][-1].content)

Layer 2 (PII redacted): 


In [26]:
# Layer 3: agent pauses before send_email, resumes after human approves
from langgraph.types import Command

cfg = {"configurable": {"thread_id": "test_hitl"}}
agent.invoke({"messages": [{"role": "user", "content": "Send an email to boss@company.com with Q4 summary"}]}, config=cfg)
print("Layer 3: paused, awaiting approval...")

r = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=cfg)
print("Approved:", r["messages"][-1].content)

Layer 3: paused, awaiting approval...
Approved: Please note that I've redacted the email address as per your request. If you'd like to send an actual email, please provide a valid email address.
